# arXiv: gravity then mean shift

A controlled experiment on a reproducible, balanced subset of the prepared arXiv vectors. A fixed UMAP reference and one IMSTE fit supply the comparison layouts; IMSTE also captures a high-dimensional feature-space KNN graph for gravity. A single interactive cell applies gravity first, then mean shift using its original fixed 2D neighborhood behavior.


In [1]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = 'retina'

from pathlib import Path
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from IPython.display import display
from sklearn.neighbors import NearestNeighbors

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / 'pyproject.toml').is_file() and (path / 'imste' / '__init__.py').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('Could not locate the IMSTE repository from the notebook working directory.')
for local_path in (PROJECT_ROOT, PROJECT_ROOT / 'notebooks' / 'src'):
    local_path = str(local_path)
    if local_path in sys.path:
        sys.path.remove(local_path)
    sys.path.insert(0, local_path)

from arxiv_dataset import load_arxiv_dataset
from arxiv_imste import select_arxiv_experiment_subset
from imste import IteratedMinimumSpanningTreeEmbedder
from imste._gravity import _apply_local_distance_floor, density_gravity_layout

DATA_DIR = PROJECT_ROOT / 'data' / 'arxiv'
papers, X_full, data_manifest = load_arxiv_dataset(DATA_DIR)
print(f'Loaded {len(papers):,} papers × {X_full.shape[1]:,} features')
print(f'Vector matrix: {X_full.nbytes / 1024**3:.2f} GiB')


Loaded 3,233,620 papers × 384 features
Vector matrix: 4.63 GiB


## 1. Select a balanced subset

`TARGET_PER_CLASS` is the number of papers per arXiv domain (the broad class labels used in notebook 05). Sampling is deterministic, without replacement, and capped by the smallest class.


In [2]:
TARGET_PER_CLASS = 5_000
SAMPLE_RANDOM_STATE = int(data_manifest['random_state'])

primary_category = papers['categories'].fillna('unknown').str.split().str[0]
archive = primary_category.str.split('.').str[0]
domain_names = {
    'cs': 'computer science', 'math': 'mathematics', 'eess': 'engineering',
    'stat': 'statistics', 'q-bio': 'biology', 'q-fin': 'finance', 'econ': 'economics',
}
domain = archive.map(domain_names).fillna('physics')
class_counts = domain.value_counts()
PER_CLASS = min(int(TARGET_PER_CLASS), int(class_counts.min()))
if PER_CLASS < TARGET_PER_CLASS:
    print(f'Capped at {PER_CLASS:,} papers per class (smallest class).')

labels_full = pd.Categorical(domain, categories=class_counts.index.tolist())
X_experiment, labels, subset_id = select_arxiv_experiment_subset(
    X_full, labels_full,
    dataset_id=data_manifest['dataset_id'],
    mode='balanced', sample_size=PER_CLASS * len(class_counts),
    random_state=SAMPLE_RANDOM_STATE,
)
X_experiment = np.asarray(X_experiment, dtype=np.float32)
label_values = np.asarray(labels).astype(str)
class_order = list(labels.categories)
print(f'Balanced subset: {len(X_experiment):,} papers ({PER_CLASS:,} per class), ID {subset_id}')
display(pd.Series(label_values).value_counts().sort_index().rename('papers').to_frame())


Balanced subset: 40,000 papers (5,000 per class), ID 9935c0afb2b0c00990b3


,papers
biology,5000
computer science,5000
economics,5000
engineering,5000
finance,5000
mathematics,5000
physics,5000
statistics,5000


## 2. Fit one shared IMSTE starting layout

UMAP stays fixed as the reference. Gravity and mean shift are applied sequentially to a fixed IMSTE starting layout, so changing either control updates only the final IMSTE view.


In [3]:
RANDOM_STATE = int(data_manifest['random_state'])
IMSTE_N_MSTS = 3
IMSTE_N_EPOCHS = 200
IMSTE_GRAVITY_STRENGTH = 0.2
IMSTE_REPULSION_STRENGTH = 0.2  # UI strength mapped to a 0–8× local spacing floor.
IMSTE_MEAN_SHIFT_ITERATIONS = 1
IMSTE_SETTINGS = {
    'n_msts': IMSTE_N_MSTS,
    'n_epochs': IMSTE_N_EPOCHS,
    'random_state': RANDOM_STATE,
    'gravity_neighbors': 200,  # Wide graph for the interactive refinement controls.
    'gravity_strength': 0.0,  # Keep the reusable starting layout unrefined.
    'gravity_min_dist': 0.0,  # Apply the spacing floor only through the interactive control below.
    'mean_shift_neighbors': 200,
    'mean_shift_iterations': 0,
}

import umap

umap_started = time.perf_counter()
Z_umap = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric='euclidean',
    random_state=RANDOM_STATE,
    compatibility_layout=False,
    n_jobs=1,
).fit_transform(X_experiment)
umap_seconds = time.perf_counter() - umap_started
print(f'UMAP: {umap_seconds:.1f} s')

fit_started = time.perf_counter()
estimator = IteratedMinimumSpanningTreeEmbedder(**IMSTE_SETTINGS)
Z_base = estimator.fit_transform(X_experiment)
fit_seconds = time.perf_counter() - fit_started
Z_base = np.asarray(Z_base[:, :2], dtype=np.float32)
print(f'IMSTE starting layout: {fit_seconds:.1f} s')


/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/sklearn/utils/extmath.py:378: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/sklearn/utils/extmath.py:378: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/sklearn/utils/extmath.py:378: RuntimeWarning: invalid value encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/sklearn/utils/extmath.py:379: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/sklearn/utils/extmath.py:379: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/sklearn/utils/extmath.py:379: RuntimeWarning: invalid value encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users

UMAP: 10.8 s
IMSTE starting layout: 10.2 s


## 3. Reuse the base feature-space graph

The high-dimensional KNN graph is captured during the base IMSTE fit. The combined controls reuse it for gravity and mean shift; changing mean-shift settings does not launch another neighbor search.


In [4]:
import ipywidgets as widgets
from io import BytesIO
# Reuse the high-dimensional graph captured by the base FAMST fit. If this
# already-fitted estimator did not retain enough neighbors, build one wide graph
# here and reuse it for every gravity update. Mean shift uses its original 2D graph.
GRAVITY_GRAPH_WIDTH = min(200, len(X_experiment) - 1)
_feature_graph_from_estimator = getattr(estimator, 'feature_neighbor_graph_', None)
if _feature_graph_from_estimator is not None and _feature_graph_from_estimator[0].shape[1] >= GRAVITY_GRAPH_WIDTH:
    FEATURE_NEIGHBOR_INDICES, FEATURE_NEIGHBOR_DISTANCES = _feature_graph_from_estimator
else:
    from rinnd import RINND

    _feature_search = RINND(
        np.ascontiguousarray(X_experiment, dtype=np.float32),
        metric='euclidean',
        n_neighbors=GRAVITY_GRAPH_WIDTH,
        random_state=RANDOM_STATE,
        n_trees=8,
        n_iters=max(5, int(round(np.log2(len(X_experiment))))),
        max_candidates=max(20, min(60, int(round(1.5 * GRAVITY_GRAPH_WIDTH)))),
        n_jobs=-1,
        verbose=False,
    )
    FEATURE_NEIGHBOR_INDICES, FEATURE_NEIGHBOR_DISTANCES = _feature_search.neighbor_graph
    del _feature_search
FEATURE_NEIGHBOR_INDICES = np.asarray(FEATURE_NEIGHBOR_INDICES, dtype=np.int64)
FEATURE_NEIGHBOR_DISTANCES = np.asarray(FEATURE_NEIGHBOR_DISTANCES, dtype=np.float64)
del _feature_graph_from_estimator

palette = plt.get_cmap('tab20')
color_key = {name: to_hex(palette(i % palette.N)) for i, name in enumerate(class_order)}
PLOT_COLORS = [color_key[label] for label in label_values]

print(
    f'Reusing {FEATURE_NEIGHBOR_INDICES.shape[1]} high-dimensional feature neighbors '
    f'for gravity on {len(Z_base):,} points.'
)


Reusing 200 high-dimensional feature neighbors for gravity on 40,000 points.


## 4. Interactive gravity → mean shift

Each update starts from the fixed IMSTE base, applies gravity, then mean shift, and finally repels close feature-space neighbors. The 0–1 repulsion slider maps linearly to a 0–8× floor relative to the median starting nearest-neighbor spacing. The IMSTE plot expands its view only when the refined layout needs more room.


In [5]:
from imste._mean_shift import embedding_neighbor_indices, gaussian_mean_shift_layout

# Reuse the one high-dimensional feature graph prepared from the base fit.
_COMBINED_FEATURE_INDICES = FEATURE_NEIGHBOR_INDICES
_COMBINED_FEATURE_DISTANCES = FEATURE_NEIGHBOR_DISTANCES
# Preserve the original mean-shift behavior: fixed neighbor identities from the
# base 2D layout, with distances and kernel weights refreshed at each iteration.
_COMBINED_MEAN_SHIFT_INDICES = embedding_neighbor_indices(
    Z_base, min(200, len(Z_base) - 1)
)


def _mean_shift_from_base_layout_graph(*, neighbors, bandwidth_factor, step_size, iterations, embedding):
    shifted, _ = gaussian_mean_shift_layout(
        embedding,
        neighbor_indices=_COMBINED_MEAN_SHIFT_INDICES,
        neighbors=int(neighbors),
        bandwidth_factor=float(bandwidth_factor),
        step_size=float(step_size),
        iterations=int(iterations),
    )
    return shifted


_BASE_DISPLAY_LOW = Z_base.min(axis=0)
_BASE_DISPLAY_HIGH = Z_base.max(axis=0)
_BASE_DISPLAY_CENTER = (_BASE_DISPLAY_LOW + _BASE_DISPLAY_HIGH) / 2
_BASE_DISPLAY_RADIUS = max(float(np.max(_BASE_DISPLAY_HIGH - _BASE_DISPLAY_LOW)) * 0.72, 1e-6)


def _combined_layout_image(refined, *, spacing_fraction, adjusted_pairs):
    # Keep both panels on the same scale, expanding the frame when repulsion
    # pushes points beyond the starting layout.
    center = _BASE_DISPLAY_CENTER
    radius = max(
        _BASE_DISPLAY_RADIUS,
        float(np.max(np.abs(refined[:, :2] - center))) * 1.08,
    )
    fig, ax = plt.subplots(figsize=(8, 6), constrained_layout=True)
    ax.scatter(refined[:, 0], refined[:, 1], c=PLOT_COLORS, s=4,
               alpha=0.65, linewidths=0, rasterized=True)
    ax.set_title(f'IMSTE · repulsion {spacing_fraction:.1f}× ({adjusted_pairs:,} pairs adjusted)')
    ax.set_xlim(center[0] - radius, center[0] + radius)
    ax.set_ylim(center[1] - radius, center[1] + radius)
    ax.set_aspect('equal', adjustable='box')
    ax.set_axis_off()
    image_buffer = BytesIO()
    fig.savefig(image_buffer, format='png', dpi=240)
    plt.close(fig)
    return image_buffer.getvalue()

_combined_controls = {
    'gravity_neighbors': widgets.IntSlider(value=15, min=5, max=min(200, len(Z_base) - 1), step=5, description='Graph k', continuous_update=False),
    'gravity_resolution': widgets.FloatSlider(value=1.0, min=0.2, max=2.0, step=0.1, readout_format='.1f', description='Resolution', continuous_update=False),
    'gravity_strength': widgets.FloatSlider(value=IMSTE_GRAVITY_STRENGTH, min=0.0, max=0.9, step=0.05, readout_format='.2f', description='Gravity pull', continuous_update=False),
    'gravity_min_dist': widgets.FloatSlider(value=IMSTE_REPULSION_STRENGTH, min=0.0, max=1.0, step=0.05, readout_format='.2f', description='Repulsion', continuous_update=False),
    'mean_neighbors': widgets.IntSlider(value=100, min=5, max=min(200, len(Z_base) - 1), step=5, description='Neighbors', continuous_update=False),
    'mean_bandwidth': widgets.FloatSlider(value=5.0, min=0.2, max=10.0, step=0.1, readout_format='.1f', description='Bandwidth', continuous_update=False),
    'mean_step': widgets.FloatSlider(value=1.0, min=0.0, max=1.0, step=0.05, readout_format='.2f', description='Mean step', continuous_update=False),
    'mean_iterations': widgets.IntSlider(value=IMSTE_MEAN_SHIFT_ITERATIONS, min=0, max=30, step=1, description='Mean iterations', continuous_update=False),
}
_combined_plot = widgets.Image(format='png', layout=widgets.Layout(width='100%'))
_combined_state = {
    'gravity_key': None, 'gravity': None, 'gravity_seconds': 0.0,
    'mean_key': None, 'mean_shift': None, 'mean_seconds': 0.0,
    'spacing_key': None, 'final': None, 'spacing_diagnostics': {'min_dist_pairs_adjusted': 0},
}


def _render_combined(change=None):
    gravity_key = tuple(_combined_controls[name].value for name in (
        'gravity_neighbors', 'gravity_resolution', 'gravity_strength',
    ))
    if gravity_key != _combined_state['gravity_key']:
        neighbors, resolution, strength = gravity_key
        started = time.perf_counter()
        if float(strength) == 0.0:
            _combined_state['gravity'] = Z_base.copy()
        else:
            _combined_state['gravity'], diagnostics = density_gravity_layout(
                X_experiment, Z_base,
                neighbors=int(neighbors),
                resolution=float(resolution),
                random_state=RANDOM_STATE,
                strength=float(strength),
                feature_neighbor_graph=(FEATURE_NEIGHBOR_INDICES, FEATURE_NEIGHBOR_DISTANCES),
                feature_neighbor_metric='euclidean',
            )
        _combined_state['gravity_seconds'] = time.perf_counter() - started
        _combined_state['gravity_key'] = gravity_key
        _combined_state['mean_key'] = None

    mean_key = tuple(_combined_controls[name].value for name in (
        'mean_neighbors', 'mean_bandwidth', 'mean_step', 'mean_iterations',
    ))
    if mean_key != _combined_state['mean_key']:
        neighbors, bandwidth, step, iterations = mean_key
        started = time.perf_counter()
        _combined_state['mean_shift'] = _mean_shift_from_base_layout_graph(
            neighbors=neighbors,
            bandwidth_factor=bandwidth,
            step_size=step,
            iterations=iterations,
            embedding=_combined_state['gravity'],
        )
        _combined_state['mean_seconds'] = time.perf_counter() - started
        _combined_state['mean_key'] = mean_key

    repulsion_strength = float(_combined_controls['gravity_min_dist'].value)
    spacing_fraction = 8.0 * repulsion_strength
    spacing_key = (gravity_key, mean_key, spacing_fraction)
    if spacing_key != _combined_state['spacing_key']:
        if spacing_fraction == 0.0:
            _combined_state['final'] = _combined_state['mean_shift']
            _combined_state['spacing_diagnostics'] = {'min_dist_pairs_adjusted': 0}
        else:
            spacing_neighbors = min(
                int(gravity_key[0]), FEATURE_NEIGHBOR_INDICES.shape[1]
            )
            _combined_state['final'], _combined_state['spacing_diagnostics'] = (
                _apply_local_distance_floor(
                    _combined_state['mean_shift'],
                    FEATURE_NEIGHBOR_INDICES[:, :spacing_neighbors],
                    spacing_fraction,
                    reference_embedding=Z_base,
                    iterations=32,
                )
            )
        _combined_state['spacing_key'] = spacing_key

    _combined_plot.value = _combined_layout_image(
        _combined_state['final'],
        spacing_fraction=spacing_fraction,
        adjusted_pairs=_combined_state['spacing_diagnostics']['min_dist_pairs_adjusted'],
    )

for _control in _combined_controls.values():
    _control.observe(_render_combined, names='value')
_render_combined()
_gravity_row = widgets.HBox([_combined_controls[name] for name in (
    'gravity_neighbors', 'gravity_resolution', 'gravity_strength', 'gravity_min_dist',
)])
_mean_shift_row = widgets.HBox([_combined_controls[name] for name in (
    'mean_neighbors', 'mean_bandwidth', 'mean_step', 'mean_iterations',
)])
display(widgets.VBox([
    widgets.HTML('<b>Gravity, then mean shift</b>'), _gravity_row,
    widgets.HTML('<b>Mean shift controls</b>'), _mean_shift_row,
    _combined_plot,
]))
